<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/07_graph_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/aml-project/'
q = pd.read_parquet(base + 'alert_queue.parquet')
q['Date'] = pd.to_datetime(q['Date'])

cutoff = pd.Timestamp('2023-06-01')
train_raw = q[q['Date'] < cutoff]

edges = train_raw.groupby(['Sender_account','Receiver_account']).size().reset_index(name='weight')
n_nodes = len(set(edges['Sender_account']) | set(edges['Receiver_account']))

print(f"Edges (unique pairs): {len(edges):,}")
print(f"Nodes (accounts):     {n_nodes:,}")
print(f"Avg edges per node:   {len(edges)/n_nodes:.2f}")
print(f"\nEdge weight distribution:")
print(edges['weight'].describe().round(2))

Mounted at /content/drive
Edges (unique pairs): 84,965
Nodes (accounts):     102,287
Avg edges per node:   0.83

Edge weight distribution:
count    84965.00
mean         1.92
std          1.23
min          1.00
25%          1.00
50%          2.00
75%          2.00
max         27.00
Name: weight, dtype: float64


In [2]:
!pip install networkx -q
import networkx as nx

G = nx.DiGraph()
G.add_weighted_edges_from(edges[['Sender_account','Receiver_account','weight']].values)

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Density: {nx.density(G):.8f}")

undirected = G.to_undirected()
comps = list(nx.connected_components(undirected))
sizes = sorted([len(c) for c in comps], reverse=True)
print(f"\nConnected components: {len(comps):,}")
print(f"Largest 5 sizes: {sizes[:5]}")
print(f"Components of size 2: {sum(1 for s in sizes if s == 2):,}")

Graph: 102,287 nodes, 84,965 edges
Density: 0.00000812

Connected components: 18,517
Largest 5 sizes: [84, 77, 72, 66, 59]
Components of size 2: 3,411


In [3]:
df = pd.read_parquet(base + 'sample.parquet')
df['Date'] = pd.to_datetime(df['Date'])
train_all = df[df['Date'] < cutoff]

print(f"Training transactions (all, not just alerts): {len(train_all):,}")

edges2 = train_all.groupby(['Sender_account','Receiver_account']).size().reset_index(name='weight')
n2 = len(set(edges2['Sender_account']) | set(edges2['Receiver_account']))

print(f"Edges: {len(edges2):,}")
print(f"Nodes: {n2:,}")
print(f"Avg edges per node: {len(edges2)/n2:.2f}")

Training transactions (all, not just alerts): 1,119,391
Edges: 536,326
Nodes: 546,375
Avg edges per node: 0.98


In [4]:
G2 = nx.DiGraph()
G2.add_weighted_edges_from(edges2[['Sender_account','Receiver_account','weight']].values)

und2 = G2.to_undirected()
comps2 = sorted([len(c) for c in nx.connected_components(und2)], reverse=True)

print(f"Nodes {G2.number_of_nodes():,} | Edges {G2.number_of_edges():,}")
print(f"Components: {len(comps2):,}")
print(f"Largest 5: {comps2[:5]}")
print(f"Largest as % of nodes: {comps2[0]/G2.number_of_nodes()*100:.1f}%")

Nodes 546,375 | Edges 536,326
Components: 19,216
Largest 5: [459, 449, 429, 401, 378]
Largest as % of nodes: 0.1%


In [5]:
gr = pd.DataFrame([
    {'graph_source':'alert queue (train)', 'transactions':163419,
     'nodes':102287, 'edges':84965, 'edges_per_node':0.83,
     'components':18517, 'largest_component':84, 'largest_pct':0.08},
    {'graph_source':'full sample (train)', 'transactions':1119391,
     'nodes':546375, 'edges':536326, 'edges_per_node':0.98,
     'components':19216, 'largest_component':459, 'largest_pct':0.08},
])
gr.to_csv(base + 'graph_feasibility.csv', index=False)
print(gr.to_string(index=False))

       graph_source  transactions  nodes  edges  edges_per_node  components  largest_component  largest_pct
alert queue (train)        163419 102287  84965            0.83       18517                 84         0.08
full sample (train)       1119391 546375 536326            0.98       19216                459         0.08
